# 00A — Check and repair corrected source-Zarr metadata

Run this **before retrying source loading in notebook 00**. Keep it beside your existing
`pilot_config.json`, `cosmx_rgb_pilot.py`, and the new `cosmx_zarr_schema_tools.py`.
Your edited config is read, not replaced.

The traceback fails before FOV layout or Cellpose. In SpatialData 0.7.2, container
format `0.1` uses legacy `multiscales` raster attributes, whereas container format
`0.2` uses `ome`. Those format numbers are **not** the installed package versions.

This notebook audits first. Applying a repair is explicit and only changes a verified
legacy root declaration and its consolidated metadata cache. Raster data, existing
image flips, label IDs, transcript coordinates, transformations, and AnnData contents
are not changed. Stop other writers to these source stores before applying it.

Sources: [reader, v0.7.2](https://raw.githubusercontent.com/scverse/spatialdata/v0.7.2/src/spatialdata/_io/io_zarr.py),
[metadata writer, v0.7.2](https://raw.githubusercontent.com/scverse/spatialdata/v0.7.2/src/spatialdata/_core/spatialdata.py).


In [1]:
from pathlib import Path
import sys
import importlib
import json
import pandas as pd

HERE = Path.cwd()  # Change only if the notebook is not in the bundle directory.
for filename in ("pilot_config.json", "cosmx_rgb_pilot.py", "cosmx_zarr_schema_tools.py"):
    if not (HERE / filename).is_file():
        raise FileNotFoundError(f"Set HERE to the bundle directory. Missing: {HERE / filename}")
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))
import cosmx_zarr_schema_tools as schema
importlib.reload(schema)
config = json.loads((HERE / "pilot_config.json").read_text())

ORIGINAL_ROOT = Path(
    "/stash/data/nonclin/TBIO-7804_CosMx6k-DS2025022748371-CRC/"
    "derived_files/batch_202506/zarr_files"
)
REFERENCE_FILES = {
    "B20317610105_colon": "B20317610105_colon_202508.zarr",
    "B20971880114_colon": "B20971880114_colon_202508.zarr",
}
SAMPLE_KEYS = list(config["samples"])
plans = {
    key: {
        "corrected": Path(config["samples"][key]["source_zarr"]),
        "reference": ORIGINAL_ROOT / REFERENCE_FILES[key],
    }
    for key in SAMPLE_KEYS
}
display(pd.DataFrame(plans).T)


,corrected,reference
B20317610105_colon,/host_root/nethome/reny28/Projects/CosMX_proje...,/stash/data/nonclin/TBIO-7804_CosMx6k-DS202502...
B20971880114_colon,/host_root/nethome/reny28/Projects/CosMX_proje...,/stash/data/nonclin/TBIO-7804_CosMx6k-DS202502...


## 1. Read-only audit

The audit reads actual on-disk metadata, ignoring the consolidated snapshot.
It checks all image/label elements, not only the first FOV.

Possible actions:
- `restore_legacy_container_0.1`: every raster is a supported legacy raster, but the root declares 0.2.
- `discard_stale_consolidated_cache`: the root is already correct; its cached metadata is outdated.
- `none`: no issue within this audit's scope.
- `stop_manual_review`: mixed formats, incomplete groups, missing/mismatched rasters, or another unsupported case.

A temporary `backup_*` group is reported, never automatically deleted. This audit checks
metadata and array descriptors, not every pixel/transcript payload chunk.


In [2]:
audits = {}
rows = []
for key, paths in plans.items():
    print("\nAUDIT:", key)
    info, raster_report = schema.audit_source_zarr(
        paths["corrected"], reference_path=paths["reference"]
    )
    audits[key] = (info, raster_report)
    rows.append({
        "sample": key,
        "root_version": info["root_container_version"],
        "rasters_expect": info["raster_expected_container"],
        "original_version": info["reference_version"],
        "zarr_on_disk": info["storage_zarr_format"],
        "images": info["n_images"],
        "labels": info["n_labels"],
        "stale_cache": info["consolidated_cache"]["stale"],
        "action": info["action"],
    })
    display(raster_report[["path", "family", "legacy_versions", "ome_version", "temporary_name", "error"]].head(8))
    if info["errors"]:
        print("REVIEW REQUIRED:\n" + "\n".join(info["errors"]))
        display(raster_report[raster_report["error"].notna() | raster_report["temporary_name"]])
    if info["consolidated_cache"]["stale"]:
        print("Stale-cache examples:", info["consolidated_cache"]["examples"])
audit_summary = pd.DataFrame(rows)
display(audit_summary)



AUDIT: B20317610105_colon


,path,family,legacy_versions,ome_version,temporary_name,error
0,images/100_image,legacy_multiscales,[0.4-dev-spatialdata],None,False,None
1,images/101_image,legacy_multiscales,[0.4-dev-spatialdata],None,False,None
2,images/102_image,legacy_multiscales,[0.4-dev-spatialdata],None,False,None
3,images/103_image,legacy_multiscales,[0.4-dev-spatialdata],None,False,None
4,images/104_image,legacy_multiscales,[0.4-dev-spatialdata],None,False,None
5,images/105_image,legacy_multiscales,[0.4-dev-spatialdata],None,False,None
6,images/106_image,legacy_multiscales,[0.4-dev-spatialdata],None,False,None
7,images/107_image,legacy_multiscales,[0.4-dev-spatialdata],None,False,None


Stale-cache examples: ['images/.zattrs (cache only)', 'images/.zgroup', 'images/100_image/.zgroup', 'images/100_image/0/.zattrs (cache only)', 'images/101_image/.zgroup', 'images/101_image/0/.zattrs (cache only)', 'images/102_image/.zgroup', 'images/102_image/0/.zattrs (cache only)', 'images/103_image/.zgroup', 'images/103_image/0/.zattrs (cache only)', 'images/104_image/.zgroup', 'images/104_image/0/.zattrs (cache only)', 'images/105_image/.zgroup', 'images/105_image/0/.zattrs (cache only)', 'images/106_image/.zgroup', 'images/106_image/0/.zattrs (cache only)', 'images/107_image/.zgroup', 'images/107_image/0/.zattrs (cache only)', 'images/108_image/.zgroup', 'images/108_image/0/.zattrs (cache only)']

AUDIT: B20971880114_colon


,path,family,legacy_versions,ome_version,temporary_name,error
0,images/100_image,legacy_multiscales,[0.4-dev-spatialdata],None,False,None
1,images/101_image,legacy_multiscales,[0.4-dev-spatialdata],None,False,None
2,images/102_image,legacy_multiscales,[0.4-dev-spatialdata],None,False,None
3,images/103_image,legacy_multiscales,[0.4-dev-spatialdata],None,False,None
4,images/104_image,legacy_multiscales,[0.4-dev-spatialdata],None,False,None
5,images/105_image,legacy_multiscales,[0.4-dev-spatialdata],None,False,None
6,images/106_image,legacy_multiscales,[0.4-dev-spatialdata],None,False,None
7,images/107_image,legacy_multiscales,[0.4-dev-spatialdata],None,False,None


Stale-cache examples: ['images/.zattrs (cache only)', 'images/.zgroup', 'images/100_image/.zgroup', 'images/100_image/0/.zattrs (cache only)', 'images/101_image/.zgroup', 'images/101_image/0/.zattrs (cache only)', 'images/102_image/.zgroup', 'images/102_image/0/.zattrs (cache only)', 'images/103_image/.zgroup', 'images/103_image/0/.zattrs (cache only)', 'images/104_image/.zgroup', 'images/104_image/0/.zattrs (cache only)', 'images/105_image/.zgroup', 'images/105_image/0/.zattrs (cache only)', 'images/106_image/.zgroup', 'images/106_image/0/.zattrs (cache only)', 'images/107_image/.zgroup', 'images/107_image/0/.zattrs (cache only)', 'images/108_image/.zgroup', 'images/108_image/0/.zattrs (cache only)']


,sample,root_version,rasters_expect,original_version,zarr_on_disk,images,labels,stale_cache,action
0,B20317610105_colon,0.2,0.1,0.1,2,200,200,True,restore_legacy_container_0.1
1,B20971880114_colon,0.2,0.1,0.1,2,200,200,True,restore_legacy_container_0.1


## 2. Explicit, one-time repair

First run with `APPLY_REPAIR=False`. After reviewing the audit and stopping other source-store
writers, set it to `True` and rerun this cell.

The helper verifies the corrected rasters against the originals, backs up the exact root
metadata/cache files in an external sibling directory, preserves all unrelated attributes,
and changes only the necessary root version field. It invalidates `.zmetadata` so a stale
snapshot cannot override the fix. No image/label arrays, table contents or transforms are rewritten.
Re-running after successful repair is a no-op. A mixed or partial store is refused.


In [3]:
APPLY_REPAIR = False  # Set True only after inspecting the previous cell.
repair_results = {}
for key, paths in plans.items():
    print("\nREPAIR PLAN:", key)
    repair_results[key] = schema.repair_source_zarr_metadata(
        paths["corrected"],
        reference_path=paths["reference"],
        apply=APPLY_REPAIR,
    )
display(pd.DataFrame([
    {"sample": key, "status": result["status"], "changed": result["changed"],
     "backup_path": result["backup_path"]}
    for key, result in repair_results.items()
]))



REPAIR PLAN: B20317610105_colon
[schema] B20317610105_colon_202508.zarr: declared=0.2, rasters expect=0.1, action=restore_legacy_container_0.1
[dry run] No source files changed. Set apply=True after reviewing the audit.

REPAIR PLAN: B20971880114_colon
[schema] B20971880114_colon_202508.zarr: declared=0.2, rasters expect=0.1, action=restore_legacy_container_0.1
[dry run] No source files changed. Set apply=True after reviewing the audit.


,sample,status,changed,backup_path
0,B20317610105_colon,dry_run,False,None
1,B20971880114_colon,dry_run,False,None


In [4]:
# now replicated and change to True for actual repair
APPLY_REPAIR = True  # Set True only after inspecting the previous cell.
repair_results = {}
for key, paths in plans.items():
    print("\nREPAIR PLAN:", key)
    repair_results[key] = schema.repair_source_zarr_metadata(
        paths["corrected"],
        reference_path=paths["reference"],
        apply=APPLY_REPAIR,
    )
display(pd.DataFrame([
    {"sample": key, "status": result["status"], "changed": result["changed"],
     "backup_path": result["backup_path"]}
    for key, result in repair_results.items()
]))



REPAIR PLAN: B20317610105_colon
[schema] B20317610105_colon_202508.zarr: declared=0.2, rasters expect=0.1, action=restore_legacy_container_0.1
[backup] /host_root/nethome/reny28/Projects/CosMX_projects/P06364_CRC_DS20250227_48371/tmp/DS-20250227-48371/B20317610105_colon_202508.zarr.schema_backup_20260916T203924Z_b76e5387
[done] Root declaration/cache repaired. Rasters, points, tables and transforms untouched.
[next] Strict-load with pilot.load_source(...). Re-consolidation is optional after validation.

REPAIR PLAN: B20971880114_colon
[schema] B20971880114_colon_202508.zarr: declared=0.2, rasters expect=0.1, action=restore_legacy_container_0.1
[backup] /host_root/nethome/reny28/Projects/CosMX_projects/P06364_CRC_DS20250227_48371/tmp/DS-20250227-48371/B20971880114_colon_202508.zarr.schema_backup_20260916T203950Z_6585f637
[done] Root declaration/cache repaired. Rasters, points, tables and transforms untouched.
[next] Strict-load with pilot.load_source(...). Re-consolidation is optional 

,sample,status,changed,backup_path
0,B20317610105_colon,metadata_repaired,True,/host_root/nethome/reny28/Projects/CosMX_proje...
1,B20971880114_colon,metadata_repaired,True,/host_root/nethome/reny28/Projects/CosMX_proje...


## 3. Verify with the original strict pilot loader

This uses the same `pilot.load_source()` that failed in notebook 00. It does not use
`on_bad_files="warn"` or omit labels. The per-FOV pairing check verifies that each image has
its corresponding labels and points and that full-resolution image/label dimensions agree.
The table is loaded normally; this is not a complete pixel/chunk integrity test.


In [5]:
import cosmx_rgb_pilot as pilot
importlib.reload(pilot)
verification_rows = []
for key, paths in plans.items():
    info, _ = schema.audit_source_zarr(paths["corrected"], reference_path=paths["reference"])
    if info["action"] != "none":
        raise RuntimeError(f"{key}: finish the audit/repair first; pending action={info['action']}")
    source = pilot.load_source(paths["corrected"])
    pairs = pilot.discover_fovs(source, config["samples"][key])
    verification_rows.append({
        "sample": key,
        "images": len(source.images),
        "labels": len(source.labels),
        "points": len(source.points),
        "tables": len(source.tables),
        "complete_fov_pairs": len(pairs),
        "status": "strict_load_ok",
    })
    del source
verification = pd.DataFrame(verification_rows)
display(verification)


/home/domino/reny28/Projects/CosMX_projects/P06364_CRC_DS20250227_48371/env/gpu_env/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/domino/reny28/Projects/CosMX_projects/P06364_CRC_DS20250227_48371/env/gpu_env/.venv/lib/python3.13/site-packages/spatialdata/_core/query/relational_query.py:531: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  left = partial(_left_join_spatialelement_table)
/home/domino/reny28/Projects/CosMX_projects/P06364_CRC_DS20250227_48371/env/gpu_env/.venv/lib/python3.13/site-packages/spatialdata/_core/query/relational_query.py:532: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve

,sample,images,labels,points,tables,complete_fov_pairs,status
0,B20317610105_colon,200,200,200,1,200,strict_load_ok
1,B20971880114_colon,200,200,200,1,200,strict_load_ok


## 4. Optional: rebuild the consolidated metadata cache after successful loading

This improves later metadata access; it is not required for correctness. It does **not**
call `sdata.write_metadata()` or change the root format declaration. Leave disabled for the
first successful pass. Do not run concurrent source-store writes during consolidation.


In [6]:
RECONSOLIDATE_AFTER_SUCCESS = False
if RECONSOLIDATE_AFTER_SUCCESS:
    import zarr
    if len(verification_rows) != len(plans):
        raise RuntimeError("Finish strict-load verification for every source first.")
    for key, paths in plans.items():
        zarr.consolidate_metadata(str(paths["corrected"]))
        info, _ = schema.audit_source_zarr(paths["corrected"], reference_path=paths["reference"])
        if info["action"] != "none":
            raise RuntimeError(f"Consolidation did not pass audit for {key}: {info}")
        check = pilot.load_source(paths["corrected"])
        print(key, "reconsolidated and strict-loaded:", len(check.images), len(check.labels))
        del check
else:
    print("Optional cache rebuild not requested.")


Optional cache rebuild not requested.


In [7]:
# rerun to actually rebuild dataset so that we don't run into this again...
RECONSOLIDATE_AFTER_SUCCESS = True
if RECONSOLIDATE_AFTER_SUCCESS:
    import zarr
    if len(verification_rows) != len(plans):
        raise RuntimeError("Finish strict-load verification for every source first.")
    for key, paths in plans.items():
        zarr.consolidate_metadata(str(paths["corrected"]))
        info, _ = schema.audit_source_zarr(paths["corrected"], reference_path=paths["reference"])
        if info["action"] != "none":
            raise RuntimeError(f"Consolidation did not pass audit for {key}: {info}")
        check = pilot.load_source(paths["corrected"])
        print(key, "reconsolidated and strict-loaded:", len(check.images), len(check.labels))
        del check
else:
    print("Optional cache rebuild not requested.")


/home/domino/reny28/Projects/CosMX_projects/P06364_CRC_DS20250227_48371/env/gpu_env/.venv/lib/python3.13/site-packages/zarr/core/group.py:3559: ZarrUserWarning: Object at zmetadata is not recognized as a component of a Zarr hierarchy.
  warnings.warn(
/home/domino/reny28/Projects/CosMX_projects/P06364_CRC_DS20250227_48371/env/gpu_env/.venv/lib/python3.13/site-packages/zarr/core/group.py:3559: ZarrUserWarning: Object at points.parquet is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


RuntimeError: Consolidation did not pass audit for B20317610105_colon: {'path': '/host_root/nethome/reny28/Projects/CosMX_projects/P06364_CRC_DS20250227_48371/tmp/DS-20250227-48371/B20317610105_colon_202508.zarr', 'root_container_version': '0.1', 'raster_expected_container': '0.1', 'storage_zarr_format': 2, 'n_images': 200, 'n_labels': 200, 'raster_families': {'legacy_multiscales': 400}, 'consolidated_cache': {'present': True, 'stale': True, 'examples': ['images/.zattrs (cache only)', 'images/.zgroup', 'images/100_image/.zgroup', 'images/100_image/0/.zattrs (cache only)', 'images/101_image/.zgroup', 'images/101_image/0/.zattrs (cache only)', 'images/102_image/.zgroup', 'images/102_image/0/.zattrs (cache only)', 'images/103_image/.zgroup', 'images/103_image/0/.zattrs (cache only)', 'images/104_image/.zgroup', 'images/104_image/0/.zattrs (cache only)', 'images/105_image/.zgroup', 'images/105_image/0/.zattrs (cache only)', 'images/106_image/.zgroup', 'images/106_image/0/.zattrs (cache onl

## Return to notebook 00

Rerun notebook 00's source-loading block. Keep the corrected paths in your existing config.
Do not run image-flip or point-repair functions again for this metadata problem.

In the **orientation-decision** cell, use `image_flip_y=False` when inspection confirms the
saved image already agrees with transcript coordinates. Determine `label_flip_y` separately;
do not assume labels and images were corrected identically. Keep approval false until reviewed.

When changing from raw sources to physically corrected sources, distinguish their revisions
(e.g. `source_revision="image_corrected_202508_v1"`). If a previous successful run cached a
layout for the raw sources, use a new pilot output root rather than bypassing its cache checks.
The root metadata repair itself does not change pixels or coordinates.

### Prevent recurrence
A broad `sdata.write_metadata()` call in SpatialData 0.7.2 defaults to writing root attributes
using the current container format. For table/transform-only changes, use the corresponding
scoped persistence operations and keep root-format rewriting explicit. The previous
`persist_changes` version must actually omit its unconditional `sdata.write_metadata()` call;
merely changing the loading code will not prevent a later persistence call from reintroducing it.

### Validation of this supplement
The helper passed 19 synthetic JSON/directory-fixture tests; this notebook is syntax-checked
and nbformat-validated. Your actual source stores and exact SpatialData/Zarr runtime were not
available here. The strict-load cell is the instance-side verification.
